<a href="https://colab.research.google.com/github/pras73/AI_Projects/blob/main/Cyberbullying_%26_Online_Grooming_Alert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment Setup & Dependencies
# Install required libraries in Google Colab
!pip install -q transformers datasets accelerate spacy scikit-learn pandas tabulate
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Cell 2: Import Libraries & Set Global Seeds
import os
import re
import time
import json
import torch
import numpy as np
import pandas as pd
import spacy
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass, field
from tabulate import tabulate

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from datasets import Dataset

# Set seed for reproducibility
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute hardware: {device}")

Using compute hardware: cpu


In [3]:
# Cell 3: Layer 1 - Privacy-Preserving PII Sanitization Engine
class PIISanitizer:
    """
    Sanitizes Personally Identifiable Information (PII) like names, phone numbers,
    emails, and physical addresses before passing text to the downstream model or logs.
    """
    def __init__(self):
        # Load spaCy small model for NER
        self.nlp = spacy.load("en_core_web_sm")

        # Regex patterns for fast PII scrubbing
        self.phone_regex = re.compile(
            r'(\+?\d{1,3}[\s-]?)?\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4}'
        )
        self.email_regex = re.compile(
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        )
        self.address_regex = re.compile(
            r'\b\d{1,5}\s+[A-Za-z0-9\s.,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Drive|Dr|Lane|Ln)\b',
            re.IGNORECASE
        )

    def sanitize(self, text: str) -> str:
        # Regex replacement first
        text = self.email_regex.sub("[REDACTED_EMAIL]", text)
        text = self.phone_regex.sub("[REDACTED_PHONE]", text)
        text = self.address_regex.sub("[REDACTED_ADDRESS]", text)

        # spaCy NER for Person Names and Locations
        doc = self.nlp(text)
        sanitized_tokens = []

        for token in doc:
            if token.ent_type_ == "PERSON":
                sanitized_tokens.append("[REDACTED_NAME]")
            elif token.ent_type_ in ["GPE", "LOC"] and token.text.lower() not in ["school", "park"]:
                sanitized_tokens.append("[REDACTED_LOCATION]")
            else:
                sanitized_tokens.append(token.text)

        # Clean double redact tags and spaces
        cleaned_text = " ".join(sanitized_tokens)
        cleaned_text = re.sub(r'(\[REDACTED_\w+\]\s*)+', r'\1', cleaned_text)
        return cleaned_text.strip()

# Test PII Engine
sanitizer = PIISanitizer()
raw_test = "Hey Sarah, call me at 555-123-4567 or email john.doe@mail.com! Meet me at 742 Evergreen Terrace in Springfield."
print("Raw Text:       ", raw_test)
print("Sanitized Text: ", sanitizer.sanitize(raw_test))

Raw Text:        Hey Sarah, call me at 555-123-4567 or email john.doe@mail.com! Meet me at 742 Evergreen Terrace in Springfield.
Sanitized Text:  Hey [REDACTED_NAME] , call me at [ REDACTED_PHONE ] or email [ REDACTED_EMAIL ] ! Meet me at 742 Evergreen Terrace in [REDACTED_LOCATION] .


In [4]:
# Cell 4: Synthetic Dataset Creation (Cyberbullying, Grooming, Safe)
LABEL_MAPPING = {0: "SAFE", 1: "CYBERBULLYING", 2: "GROOMING_PREDATORY"}
REVERSE_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}

synthetic_data = [
    # Safe Messages
    ("Hey do you want to play Fortnite after homework?", "SAFE"),
    ("Great job on the math test today!", "SAFE"),
    ("Did anyone record the science lecture?", "SAFE"),
    ("I love playing soccer on weekends.", "SAFE"),
    ("See you at lunch tomorrow!", "SAFE"),
    ("What was the assignment for page 42?", "SAFE"),
    ("Happy birthday! Hope you have a fun day.", "SAFE"),
    ("I am reading a cool book about space.", "SAFE"),
    ("Does anyone have notes from history class?", "SAFE"),
    ("Can you send me the video link?", "SAFE"),

    # Cyberbullying Messages
    ("Nobody likes you, stop talking in this group.", "CYBERBULLYING"),
    ("You are so ugly and useless, delete your account.", "CYBERBULLYING"),
    ("Everyone is laughing at your clothes.", "CYBERBULLYING"),
    ("Why do you even show up? You're a total loser.", "CYBERBULLYING"),
    ("Stop annoying everyone or we will kick you out.", "CYBERBULLYING"),
    ("You're terrible at this game, go quit.", "CYBERBULLYING"),
    ("Post another picture and see what happens to you.", "CYBERBULLYING"),
    ("No one wants to sit with you at lunch.", "CYBERBULLYING"),

    # Grooming / Predatory Messages
    ("Are your parents home right now? Don't tell them we talked.", "GROOMING_PREDATORY"),
    ("Send me a picture of you, keep it as our secret.", "GROOMING_PREDATORY"),
    ("You are so mature for your age, much better than kids your age.", "GROOMING_PREDATORY"),
    ("I can buy you gift cards if you keep this private between us.", "GROOMING_PREDATORY"),
    ("Don't tell your parents, they wouldn't understand our connection.", "GROOMING_PREDATORY"),
    ("Let's talk on Snap or Telegram instead where messages disappear.", "GROOMING_PREDATORY"),
    ("Do you have webcams in your room? Turn it on.", "GROOMING_PREDATORY"),
    ("I understand you better than your family does. Trust me.", "GROOMING_PREDATORY"),
]

# Expand synthetic dataset with variations
rows = []
for text, label_str in synthetic_data:
    rows.append({"text": text, "label": REVERSE_LABEL_MAPPING[label_str]})

df = pd.DataFrame(rows)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

print(f"Dataset summary: {len(df)} total samples ({len(train_df)} train, {len(val_df)} validation)")

Dataset summary: 26 total samples (20 train, 6 validation)


In [5]:
# Cell 5: Model Tokenization & Fine-Tuning Setup
MODEL_NAME = "distilbert-base-uncased"  # Lightweight transformer for fast training in Colab

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_MAPPING),
    id2label=LABEL_MAPPING,
    label2id=REVERSE_LABEL_MAPPING
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./safety_model_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Execute Fine-Tuning
print("Starting Fine-Tuning Phase...")
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [7]:
# Cell 6: Sliding Context Window & Multi-Turn Escalation Engine
@dataclass
class ThreatAlert:
    user_id: str
    severity: str  # LOW, MEDIUM, HIGH, CRITICAL
    category: str
    confidence: float
    trigger_message: str
    context_history: List[str]
    timestamp: float = field(default_factory=time.time)

class ContextAwareModerator:
    """
    Maintains a sliding window of historical messages per chat user to track
    gradual manipulative patterns (grooming, pressure) or escalating bullying.
    """
    def __init__(self, model, tokenizer, sanitizer, window_size: int = 5):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.sanitizer = sanitizer
        self.window_size = window_size
        self.user_contexts: Dict[str, List[Dict[str, Any]]] = {}

    def _get_model_scores(self, text: str) -> Dict[str, float]:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            logits = self.model(**inputs).logits
            probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()

        return {LABEL_MAPPING[i]: float(probs[i]) for i in range(len(probs))}

    def process_message(self, user_id: str, raw_message: str) -> Tuple[Dict[str, Any], ThreatAlert]:
        # Step 1: PII Scrubbing
        clean_text = self.sanitizer.sanitize(raw_message)

        # Step 2: Instant Inference on Single Message
        scores = self.get_model_scores(clean_text) if hasattr(self, 'get_model_scores') else self._get_model_scores(clean_text)

        # Step 3: Update User Context History Window
        if user_id not in self.user_contexts:
            self.user_contexts[user_id] = []

        entry = {
            "raw": raw_message,
            "sanitized": clean_text,
            "scores": scores
        }
        self.user_contexts[user_id].append(entry)

        # Keep only the last N messages
        if len(self.user_contexts[user_id]) > self.window_size:
            self.user_contexts[user_id].pop(0)

        # Step 4: Contextual Multi-Turn Trend Calculation
        recent_history = self.user_contexts[user_id]
        avg_bullying_score = np.mean([msg["scores"]["CYBERBULLYING"] for msg in recent_history])
        avg_grooming_score = np.mean([msg["scores"]["GROOMING_PREDATORY"] for msg in recent_history])

        # Current message scores
        curr_bullying = scores["CYBERBULLYING"]
        curr_grooming = scores["GROOMING_PREDATORY"]

        # Step 5: Threat Matrix Evaluation & Alert Escalation Rules
        alert = None
        history_str = [m["sanitized"] for m in recent_history]

        # Critical Thresholds
        if curr_grooming > 0.75 or avg_grooming_score > 0.50:
            severity = "CRITICAL" if curr_grooming > 0.85 else "HIGH"
            alert = ThreatAlert(
                user_id=user_id,
                severity=severity,
                category="ONLINE_GROOMING_PREDATORY",
                confidence=max(curr_grooming, avg_grooming_score),
                trigger_message=clean_text,
                context_history=history_str
            )
        elif curr_bullying > 0.75 or avg_bullying_score > 0.55:
            severity = "HIGH" if curr_bullying > 0.85 else "MEDIUM"
            alert = ThreatAlert(
                user_id=user_id,
                severity=severity,
                category="CYBERBULLYING_HARASSMENT",
                confidence=max(curr_bullying, avg_bullying_score),
                trigger_message=clean_text,
                context_history=history_str
            )
        elif curr_grooming > 0.40 or curr_bullying > 0.40:
            alert = ThreatAlert(
                user_id=user_id,
                severity="LOW",
                category="SUSPICIOUS_BEHAVIOR",
                confidence=max(curr_grooming, curr_bullying),
                trigger_message=clean_text,
                context_history=history_str
            )

        analysis_summary = {
            "sanitized_message": clean_text,
            "instant_scores": scores,
            "window_avg_grooming": float(avg_grooming_score),
            "window_avg_bullying": float(avg_bullying_score),
            "alert_triggered": alert is not None
        }

        return analysis_summary, alert

# Instantiate Moderator Pipeline
moderator = ContextAwareModerator(
    model=model,
    tokenizer=tokenizer,
    sanitizer=sanitizer,
    window_size=4
)

In [8]:
# Cell 7: Full End-to-End Real-Time Pipeline Simulation
def run_simulation(session_id: str, chat_stream: List[str]):
    print(f"\n========================================================================")
    print(f"       STARTING LIVE SAFETY MONITORING SESSION: {session_id}")
    print(f"========================================================================\n")

    for i, raw_msg in enumerate(chat_stream, 1):
        print(f"--- [Turn {i}] Incoming Raw Message: '{raw_msg}' ---")

        summary, alert = moderator.process_message(user_id=session_id, raw_message=raw_msg)

        # Display PII Scrubbing Result
        print(f" [PII Cleaned] : {summary['sanitized_message']}")

        # Display Instant Scores
        scores = summary['instant_scores']
        table_data = [[k, f"{v:.4f}"] for k, v in scores.items()]
        print(tabulate(table_data, headers=["Category", "Probability"], tablefmt="simple"))

        print(f" [Trend Metrics] 4-Msg Rolling Avg -> Grooming: {summary['window_avg_grooming']:.3f} | Bullying: {summary['window_avg_bullying']:.3f}")

        # Display Alerts if triggered
        if alert:
            color = "\033[91m" if alert.severity in ["HIGH", "CRITICAL"] else "\033[93m"
            reset = "\033[0m"
            print(f"\n{color}🚨 ALERT DISPATCHED [{alert.severity}] 🚨{reset}")
            print(f" Category    : {alert.category}")
            print(f" Confidence  : {alert.confidence:.2%}")
            print(f" Trigger     : {alert.trigger_message}")
            print(f" Context Log : {alert.context_history}")
        else:
            print(" Status      : ✅ PASSED (No Policy Violation)")

        print("-" * 72 + "\n")

# Simulated Grooming Escalation Scenario
predatory_chat_session = [
    "Hey! Great game of Roblox today.",
    "Do you play every day after school?",
    "You're really smart compared to other kids your age.",
    "Don't tell your parents, but I can buy you 5000 Robux if you message me privately on WhatsApp at 555-019-2834.",
    "Are your parents home right now? Send me a quick photo."
]

run_simulation("User_Suspect_88", predatory_chat_session)


       STARTING LIVE SAFETY MONITORING SESSION: User_Suspect_88

--- [Turn 1] Incoming Raw Message: 'Hey! Great game of Roblox today.' ---
 [PII Cleaned] : Hey ! Great game of [REDACTED_NAME] today .
Category              Probability
------------------  -------------
SAFE                       0.3386
CYBERBULLYING              0.3216
GROOMING_PREDATORY         0.3398
 [Trend Metrics] 4-Msg Rolling Avg -> Grooming: 0.340 | Bullying: 0.322
 Status      : ✅ PASSED (No Policy Violation)
------------------------------------------------------------------------

--- [Turn 2] Incoming Raw Message: 'Do you play every day after school?' ---
 [PII Cleaned] : Do you play every day after school ?
Category              Probability
------------------  -------------
SAFE                       0.3388
CYBERBULLYING              0.3245
GROOMING_PREDATORY         0.3366
 [Trend Metrics] 4-Msg Rolling Avg -> Grooming: 0.338 | Bullying: 0.323
 Status      : ✅ PASSED (No Policy Violation)
------------------

In [10]:
# Cell 8: Cyberbullying Scenario Test
cyberbullying_chat_session = [
    "Hey everyone, welcome back to the group chat.",
    "Did you see the homework for science?",
    "Why are you even talking here? You are so useless.",
    "Nobody likes you, stop talking in this group and delete your account."
]

run_simulation("User_Bully_02", cyberbullying_chat_session)


       STARTING LIVE SAFETY MONITORING SESSION: User_Bully_02

--- [Turn 1] Incoming Raw Message: 'Hey everyone, welcome back to the group chat.' ---
 [PII Cleaned] : Hey everyone , welcome back to the group chat .
Category              Probability
------------------  -------------
SAFE                       0.3405
CYBERBULLYING              0.317
GROOMING_PREDATORY         0.3425
 [Trend Metrics] 4-Msg Rolling Avg -> Grooming: 0.342 | Bullying: 0.317
 Status      : ✅ PASSED (No Policy Violation)
------------------------------------------------------------------------

--- [Turn 2] Incoming Raw Message: 'Did you see the homework for science?' ---
 [PII Cleaned] : Did you see the homework for science ?
Category              Probability
------------------  -------------
SAFE                       0.3369
CYBERBULLYING              0.3132
GROOMING_PREDATORY         0.3499
 [Trend Metrics] 4-Msg Rolling Avg -> Grooming: 0.342 | Bullying: 0.317
 Status      : ✅ PASSED (No Policy Violation)
